# Project - Airline AI Assistant

We'll now bring together what we've learned to make an AI Customer Support assistant for an Airline

In [ ]:
# imports

import os
import json
from dotenv import load_dotenv
from openai import OpenAI
import gradio as gr
import sqlite3

import asyncio
from playwright.async_api import async_playwright

asyncio.set_event_loop_policy(asyncio.WindowsProactorEventLoopPolicy())

In [ ]:
# Initialization

load_dotenv(override=True)

openai_api_key = os.getenv('OPENAI_API_KEY')
if openai_api_key:
    print(f"OpenAI API Key exists and begins {openai_api_key[:8]}")
else:
    print("OpenAI API Key not set")
    
MODEL = "gpt-4.1-mini"
openai = OpenAI()

DB = "prices.db"

# Exercises and Business Applications

Add in more tools - perhaps to simulate actually booking a flight. A student has done this and provided their example in the community contributions folder.

Next: take this and apply it to your business. Make a multi-modal AI assistant with tools that could carry out an activity for your work. A customer support assistant? New employee onboarding assistant? So many possibilities! Also, see the week2 end of week Exercise in the separate Notebook.

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/thankyou.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#090;">I have a special request for you</h2>
            <span style="color:#090;">
                My editor tells me that it makes a HUGE difference when students rate this course on Udemy - it's one of the main ways that Udemy decides whether to show it to others. If you're able to take a minute to rate this, I'd be so very grateful! And regardless - always please reach out to me at ed@edwarddonner.com if I can help at any point.
            </span>
        </td>
    </tr>
</table>

In [ ]:
system_message = """
You are a helpful assistant for a scrap car dealership called Ashley's crap.
Give short, courteous answers, no more than 1 sentence.
You'll need to obtain the car registration number from the client, once you do,
you can use your tools to call the functions and provide a price.
Always be accurate. If you don't know the answer, say so.

The number plate should be of a suitable format (lower or uppercase letters are fine). If it isn't, say so

if the calculated price comes out at zero (0), tell them the car couln't be found and to check the reg
"""

In [ ]:
def calculate_price(kerb_wt: int):

    return 1.79*kerb_wt/10

In [ ]:
async def get_kerb_weight(number_plate: str) -> str:
    async with async_playwright() as p:
        browser = await p.chromium.launch(headless=True)
        page = await browser.new_page()

        try:
            await page.goto("https://www.parkers.co.uk/car-specs/", wait_until="domcontentloaded")

            await page.wait_for_timeout(1100)

            cookie_frame = page.frame(url=lambda url: "cmp.parkers.co.uk" in url)

            while cookie_frame is None:
                await page.wait_for_timeout(100)
                cookie_frame = page.frame(url=lambda url: "cmp.parkers.co.uk" in url)

            await cookie_frame.get_by_role("button", name="Agree").click()
            # Fill the correct registration input
            registration_input = page.locator(".vrm-lookup__input:visible")
      
            await registration_input.fill(number_plate)
            await page.get_by_role("button", name="Go").click()

            try:
                await page.get_by_text("Just curious", exact=True).click(timeout=28000)
            except Exception:
                return 0
            #await page.get_by_text("Just curious", exact=True).click()

            await page.locator("a.specs-confirmation__actions__navigate-button").click()

            weight = await page.locator(
             "//span[contains(@class,'specs-detail-table__item__label') and normalize-space()='Weight']/following-sibling::span[contains(@class,'specs-detail-table__item__value')]"
            ).inner_text()

          #  print("Weight:", weight)
            return weight.replace("kg", "").strip()
            
        finally:
            await browser.close()


In [ ]:
import asyncio
import threading

def run_in_proactor(coro):
    result = []
    error = []

    def runner():
        try:
            loop = asyncio.ProactorEventLoop()
            asyncio.set_event_loop(loop)
            result.append(loop.run_until_complete(coro))
        except Exception as e:
            error.append(e)
        finally:
            loop.close()

    thread = threading.Thread(target=runner)
    thread.start()
    thread.join()

    if error:
        raise error[0]

    return result[0]

In [ ]:
weight = run_in_proactor(get_kerb_weight("EJ11VYU") )  # 
print(weight)

In [ ]:
def get_price_for_vehicle(number_plate: str) -> str: 
    '''
    This function retuns the scrap value value of the car

    Args:
        number_plate (str) this should be of UK number plate format

    Returns:
        price (str): returns the price in GBPs of the scrap value of the car
    '''  
    # Step 1: get kerb weight
    kerb_weight = run_in_proactor(get_kerb_weight(number_plate) )

    # Step 2: calculate price using the weight
    price = calculate_price(int(kerb_weight))

    return f'£ + {price}'

In [ ]:
vehicle_price_function = {
    "name": "get_price_for_vehicle",
    "description": (
        "Get the price for a vehicle using its registration number."
        "First looks up the vehicle's kerb weight, then calculates the price based on that weight."
        "if the function returns None, tell the user to please try again and check the number plate"
        ),
    "parameters": {
        "type": "object",
        "properties": {
            "number_plate": {
                "type": "string",
                "description": "The vehicle registration number."
            }
        },
        "required": ["number_plate"],
        "additionalProperties": False
    }
}

tools = [
    {"type": "function", "function": vehicle_price_function}
]

In [ ]:
MODEL = "gpt-4.1-mini"

def chat(message, history):

    history = [{"role": h["role"], "content": h["content"]} for h in history]

    messages = ([{"role": "system", "content": system_message}] + history + [{"role": "user", "content": message}] )

    response = openai.chat.completions.create(model=MODEL, messages=messages, tools=tools)

    while response.choices[0].finish_reason == "tool_calls":
        message = response.choices[0].message
        responses = handle_tool_calls(message)
        messages.append(message)
        messages.extend(responses)
        response = openai.chat.completions.create(
            model=MODEL, messages=messages, tools=tools)

    return response.choices[0].message.content


In [ ]:
available_functions = {
    "get_price_for_vehicle": get_price_for_vehicle
}


def handle_tool_calls(message):
    responses = []

    for tool_call in message.tool_calls:
        function_name = tool_call.function.name
        arguments = json.loads(tool_call.function.arguments)

        function = available_functions[function_name]
        result = function(**arguments)

        responses.append({
            "role": "tool",
            "tool_call_id": tool_call.id,
            "content": str(result)
        })

    return responses

In [ ]:
gr.ChatInterface(fn=chat, type="messages").launch()

# a number plate you can try is EJ11 VYU